In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
df = pd.read_csv('data/premium.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [52]:
# 중복체크
df[df.duplicated(keep=False)]

,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [53]:
df = df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1337 non-null   int64  
 1   sex       1337 non-null   str    
 2   bmi       1332 non-null   float64
 3   children  1337 non-null   int64  
 4   smoker    1337 non-null   str    
 5   region    1337 non-null   str    
 6   charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


In [54]:
#bmi의 널처리
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [55]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder

y = df['charges']
X = df.drop('charges', axis=1)

# X = df.iloc[:, :-1]
# y = df.iloc[:, -1]

# 문자열 데이터의 수치화 > LabelEncoder
# 1. Dictionary Mapping 이용 : Label 지정
mapping = {
    'sex': {'female': 0, 'male': 1},
    'smoker': {'no': 0, 'yes': 1}
}

for col, m in mapping.items():
    df[col] = df[col].str.strip() # 공백 삭제
    X[col] = df[col].map(m)  

# region만 따로 처리 (순서가 중요하지 않다면)
X['region'] = LabelEncoder().fit_transform(df['region'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling for better performance : Only for Quanitity Attributes
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled['bmi'] = scaler.fit_transform(X_train[['bmi']]) # DataFrame 으로 보이게
X_test_scaled['bmi'] = scaler.transform(X_test[['bmi']])

In [57]:
X_train_scaled.head()

,age,sex,bmi,children,smoker,region
1114,23,1,-0.999446,0,0,0
968,21,1,-0.794559,2,0,0
599,52,0,1.159756,2,0,1
170,63,1,1.814235,0,0,2
275,47,0,-0.652713,2,0,0


# 모델별 학습 및 평가

# 선형회귀모델

In [58]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [77]:
# 선형회귀모델 : 정의 및 학습
model_linear = LinearRegression()
model_linear.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [73]:
def get_model_performance(pred, real) -> str :
    
    rmse = np.sqrt(mean_squared_error(real, pred))
    r2 = r2_score(real, pred)
    
    print(f"R-Square Score : {r2:.3f}")
    print(f"Real(Test) Value('charges') Mean : {real.mean():,.1f}, Mean-Square-Root Error : {rmse:,.1f}")
    
    return r2, rmse 

In [78]:
y_pred = model_linear.predict(X_test)

mae = mean_absolute_error(y_pred, y_test)

r2, rmse = get_model_performance(y_pred, y_test)

mae, float(rmse), r2

R-Square Score : 0.807
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 5,962.4


(4170.6423560015255, 5962.393019338356, 0.8065362865570329)

# 다항회귀모델  


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# features = PolynomialFeatures(degree=2, include_bias=False) # 2차 항까지 사용
# model_poly = Pipeline([('poly', features), ('linear', LinearRegression())])

# model_poly.fit(X_train, y_train)

for degree in [2,3,4] :
    features = PolynomialFeatures(degree=degree, include_bias=False)
    model_poly = Pipeline([('poly', features), ('linear', LinearRegression())])
    
    model_poly.fit(X_train, y_train)
    poly_pred = model_poly.predict(X_test)
    
    print("Performance at DEGREE :", degree)
    get_model_performance(poly_pred, y_test)
    print("-"*100)


Performance at DEGREE : 2
R-Square Score : 0.886
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 4,569.4
----------------------------------------------------------------------------------------------------
Performance at DEGREE : 3
R-Square Score : 0.877
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 4,745.7
----------------------------------------------------------------------------------------------------
Performance at DEGREE : 4
R-Square Score : 0.855
Real(Test) Value('charges') Mean : 14,272.0, Mean-Square-Root Error : 5,166.3
----------------------------------------------------------------------------------------------------


** 2차항에서 가장 성능이 높게 나타남 : 차수가 높아질수록 과적합 발생